In [2]:
import sys
import os
import pandas as pd


sys.path.append(os.path.abspath('..')) # Go up one level to the project root and add it to the path
from src.utils.config import get_data_path, PROJECT_ROOT
from src.utils.viz import plot_zip_population_pivot

# snippet to skip a cell using %%skip TRUE
from IPython.core.magic import register_cell_magic
from IPython import get_ipython

@register_cell_magic
def skip(line, cell):
    # If the line says 'True', we skip. Otherwise, we run it.
    should_skip = eval(line) if line else True
    
    if should_skip:
        print("Cell skipped.")
        return
    else:
        get_ipython().run_cell(cell)

## SaTScan Required Variables

#### 1. Case File

| Variable | Description | Requirement |
| --- | --- | --- |
| **Location ID** | e.g., FIPS, ZIP | Required |
| **Number of Cases** | e.g., 1 | Required |
| **Date** | Event timeline | Required |
| **Age Group** | For stratification | Optional |

---

#### 2. Population File

> *Note: Required for Poisson model; skip if using Space-Time Permutation.*

| Variable | Description | Requirement |
| --- | --- | --- |
| **Location ID** | FIPS | Required |
| **Year** | Timeline | Required |
| **Population Count** | Total count | Required |
| **Age Group** | Must match Case File | Optional |

---

#### 3. Coordinates File

| Variable | Description | Requirement |
| --- | --- | --- |
| **Location ID** | FIPS | Required |
| **Latitude** | Or X coordinate | Required |
| **Longitude** | Or Y coordinate | Required |

---

#### NOTE:
ZCTAs (ZIP Code Tabulation Areas): the Census Bureau's geographic approximations of ZIP codes

## 1. Case File 
NVDRS dataset 2023 `keyes_973_nvdrs_2023.csv`
It already contains all the required variables

## 2. Popultation File 

### Resolving 2010-2011 Denominator Shocks

**Observation:** Calculating the year-over-year percentage change between the 2010 Decennial headcount and the 2011 ACS 5-Year estimates reveals over 2,000 ZIP codes with massive, artifactual population swings (many exceeding 70-100% variance). 

**The Cause:** These artifacts are heavily concentrated in areas with Group Quarters (military bases, universities, prisons). The 2010 Decennial Census executed an exact, single-day headcount of these facilities. The ACS, however, relies on continuous rolling samples and historically struggles to capture group quarters accurately. Mixing a precise 1-day count (2010) with a 60-month rolling average (2011) creates a severe structural break in the denominator. 

**Resolution:** To prevent these artificial denominator drops from triggering false-positive spatial clusters in the SaTScan Poisson model, we will drop the 2010 Decennial data. Instead, we will backfill 2010 using the 2011 ACS 5-Year estimates. Because the 2011 ACS represents survey data collected from 2007–2011, it mathematically covers 2010 and maintains standard errors consistently across our time series.

### Harmonizing Spatiotemporal Boundaries (2010-2019 to 2020 ZCTAs)

The U.S. Census Bureau redraws ZIP Code Tabulation Area (ZCTA) boundaries every 10 years. Between 2019 and 2020, many ZCTAs were split, merged, or eliminated. If we feed unadjusted pre-2020 population data into SaTScan alongside post-2020 data, the Poisson model will interpret these boundary redraws as sudden demographic shifts. A ZCTA shrinking in landmass would show an artificial population drop, which SaTScan would misinterpret as a massive spike in the suicide rate, triggering a false-positive space-time cluster.

We must project all 2010-2019 population estimates into the modern 2020 ZCTA boundaries. 

1. We use the **Official 2010 to 2020 ZCTA Relationship File** (`tab20_zcta510_zcta520_natl.txt`).
2. We calculate an **Allocation Factor (AF)** for every boundary intersection by dividing the shared land area (`AREALAND_PART`) by the total land area of the original 2010 ZCTA (`AREALAND_ZCTA5_10`).
3. We multiply the 2010-2019 ZCTA populations by this factor and re-aggregate them under the new 2020 ZCTA (`GEOID_ZCTA5_20`) identifiers.

***Variables used to harmonize:***
| Variable | Description |
| --- | --- |
| **GEOID_ZCTA5_10** | Geographic Identifier for the 2010 ZCTA (the old 5-digit ZIP code used for merging historical data). |
| **GEOID_ZCTA5_20** | Geographic Identifier for the 2020 ZCTA (the new 5-digit ZIP code used for the final aggregation). |
| **AREALAND_ZCTA5_10** | Total land area of the 2010 ZCTA. Used as the denominator to calculate the Allocation Factor. |
| **AREALAND_PART** | Land area of the geographic intersection between the 2010 and 2020 ZCTAs. Used as the numerator to calculate the Allocation Factor. |

## 3. Coordinates File - ZCTA
#### Gazetteer latitude and longitude for ZIP codes for midpoint (2022)
ZCTA boundaries change rarely (mostly during the decennial census). Even if a boundary slightly adjusts, the shift in the centroid is microscopic compared to the radius of a spatiotemporal cluster. Will use the midpoint year coordinates for the whole period.

https://www.census.gov/geographies/reference-files/time-series/geo/gazetteer-files.html

In [ ]:
#TODO comapre pop_zip and pop_zip_c to verify that the armonizaion do something

## Execute SaTScan GUI data preparation Pipeline:

In [ ]:
%%skip True #run only once

from src.geospatial.prep_satscan import prep_satscan_gui

# 1. Define minimum required NVDRS columns (plus basic demographics)
#nvdrs_head = load_nvdrs(file_key="nvdrs", data_folder="raw", nrows=10000 )
nvdrs_columns = [
    'IncidentID', 
    'DeathDate', 
    'InjuryZip', 
    'ResidenceZip', 
    'IncidentCategory_c', 
    'PersonType', 
    'Sex', 
    'AgeYears_c'
]

# 2. Define Census ACS variables to fetch (ZCTA identifier and Total Population)
# WARNING: these two are the one that do not change name over time; age group population vars do change. 
#           
population_variables = {
    "NAME": "ZTCA5", 
    "DP05_0001E": "Population"
}

# 3. Define the year range for ACS data (2011 to 2023)
# Note: 2010 is handled internally by backfilling 2011 to avoid API shocks
population_years = range(2011, 2024)

# 4. Generate the Case, Population, and Coordinates CSVs
prep_satscan_gui(
    nvdrs_cols=nvdrs_columns,
    pop_vars=population_variables,
    pop_years=population_years,
    use_res_zip = False,
    nickname= "nores"
)


printed on Jul 10 (`use_res_zip = True`):
```
Preparing global SaTScan files...
Processing Case Data...
Processing Population Data...
Processing Coordinates Data...
Validating Coordinate Coverage...
  ↳ Triggering Healer: 1626 global ZIPs missing from Census Gazetteer.
  ↳ Successfully recovered 749 coordinates.
Enforcing strict SaTScan referential integrity...
  ↳ Analyzing 877 unmappable ZIPs with associated cases...
  ↳ Saved missingness analysis to 'dropped_unmappable_cases.csv'.
  ↳ Dropped 1113 total cases occurring in unmappable ZIPs.
Exporting mathematically validated artifacts...
```

printed on Jul 13 (`use_res_zip = False`, `nickname= "nores"`):
```
Preparing global SaTScan files...
Processing Case Data...
Processing Population Data...
Processing Coordinates Data...
Validating Coordinate Coverage...
  ↳ Triggering Healer: 1561 global ZIPs missing from Census Gazetteer.
  ↳ Successfully recovered 714 coordinates.
Enforcing strict SaTScan referential integrity...
  ↳ Analyzing 847 unmappable ZIPs with associated cases...
  ↳ Saved missingness analysis to 'nores_dropped_unmappable_cases.csv'.
  ↳ Dropped 2182 total cases occurring in unmappable ZIPs or zero-population ZIP-years.
Exporting mathematically validated artifacts...
```


### CHECK Dropped ZIPs
Location Logic applied: Prefer Injury location, fallback to Residence if missing

In [ ]:
#%%skip True
dropped_df = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "satscan" / "nores_dropped_unmappable_cases.csv")

# See what types of ZIPs were dropped
display(dropped_df['Type'].value_counts())

# Look at the worst offenders (highest case counts dropped)
display(dropped_df.head(10))

# Calculate the total percentage of cases lost to PO Boxes/Unique ZIPs
total_dropped = dropped_df['Cases'].sum()
print(f"Total cases lost to unmappable geography: {total_dropped}")

Print on Jul 10 for `use_res_zip = True`

Dropped ZIP types:
| ZIP Type | count |
| :--- | :--- |
| Unknown | 871 |
| MILITARY | 5 |
| UNIQUE | 1 |


Cases from dropped ZIPs (head):
| ZIP | dropped cases | Type | City | State |
| :--- | :--- | :--- | :--- | :--- |
| 9999 | 140 | Unknown | Unknown | Unknown |
| 88888 | 47 | UNIQUE | North Pole | DC |
| 1 | 7 | Unknown | Unknown | Unknown |
| 6999 | 6 | Unknown | Unknown | Unknown |
| 4067 | 5 | nknown | Unknown | Unknown |
| 29 | 3 | Unknown | Unknown | Unknown |
| 18525 | 3 | Unknown | Unknown | Unknown |
| 43063 | 2 | Unknown | Unknown | Unknown |
| 28165 | 2 | Unknown | Unknown | Unknown |
| 46640 | 2 | Unknown | Unknown | Unknown |

------------------------------------------------


Print on Jul 13 for `use_res_zip = False`

Dropped ZIP types:
| ZIP Type | count |
| :--- | :--- |
| Unknown | 841 |
| MILITARY | 5 |
| UNIQUE | 1 |


Cases from dropped ZIPs (head):
| ZIP | dropped cases | Type | City | State |
| :--- | :--- | :--- | :--- | :--- |
| 9999 | 138 | Unknown | Unknown | Unknown |
| 88888 | 33 | UNIQUE | North Pole | DC |
| 1 | 7 | Unknown | Unknown | Unknown |
| 6999 | 6 | Unknown | Unknown | Unknown |
| 4067 | 5 | nknown | Unknown | Unknown |
| 29 | 3 | Unknown | Unknown | Unknown |
| 18525 | 3 | Unknown | Unknown | Unknown |
| 43063 | 2 | Unknown | Unknown | Unknown |
| 28165 | 2 | Unknown | Unknown | Unknown |
| 46640 | 2 | Unknown | Unknown | Unknown |

---

# SaTScan — Run 1 (preliminary)
### Overview of Current Run Settings
critical settings that drove the preliminary analysis:
*   **Analysis Type:** Retrospective Space-Time (`AnalysisType=3`).
*   **Model:** Discrete Poisson (`ModelType=0`).
*   **Temporal Limits:** Time aggregated by Month (`TimeAggregationUnits=2`), with a maximum temporal size of 3 months (`MaxTemporalSize=3`).
*   **Spatial Limits:** Maximum spatial size set to 1% of the population at risk (`MaxSpatialSizeInPopulationAtRisk=1`). The physical distance restriction was disabled (`UseDistanceFromCenterOption=n`).

### Output issues
*   **Spatial Inflation:** Without a distance restriction, the 1% population threshold acts as the sole cap. Since 1% of the US population is roughly 3.3 million people, the algorithm must expand the circular scan window enormously in sparse rural areas just to hit this threshold. This results in massive, meaningless geographic radii that span across state lines. 
*   **Temporal Censoring:** The 3-month constraint (`MaxTemporalSize=3`) is abruptly truncating the clusters. Suicide clusters generally unfold over longer periods, meaning the current analysis is missing the true temporal span of these events.
*   **P-Value Approximations:** The extremely low p-values (e.g., 1e-17) indicate that SaTScan is utilizing the Gumbel approximation, which is expected behavior for highly significant clusters when using the default p-value reporting setting.

### parameters from Platt et al. (2022)
The Platt et al. (2022) paper utilized a 150 km maximum radius and a 15-month temporal window. It is important to note that these parameters were not derived from a biological or fixed mechanistic understanding of suicide contagion. Instead, they were pragmatic decisions driven by statistical power considerations: youth suicide is a rare event, and these windows were necessary to capture sufficient sample sizes for reliable estimates at the county level. Moving to ZIP-level resolution does not mean these parameters are invalid. In fact, a 150 km spatial cap encompasses thousands of ZCTAs and functions as an excellent upper bound. The higher spatial granularity of ZIP codes is actually beneficial, as it allows the model to detect smaller, localized clusters that might otherwise be diluted at the county level.

---

# SaTScan — Run 2

### Setup (vs. run 1)
- `UseDistanceFromCenterOption=y`, `MaxSpatialSizeInDistanceFromCenter=150` (was: population-% only, no distance cap) Relying solely on a population percentage distorts boundaries in rural versus urban areas. (150 months used in Platt et al.)
- `MaxSpatialSizeInPopulationAtRisk=50` (back to default; now a non-binding backstop)
- `MaxTemporalSize=15` months (was 3) (15 months used in Platt et al.)
- `CriteriaForReportingSecondaryClusters=0` (NoGeoOverlap, unchanged)

### Run Results analysis `nores_fullCLI_out.txt`
The model is reporting heavily overlapping clusters. Not finding 500 unique outbreaks; the algorithm is finding the exact same massive hot-zones repeatedly by slightly shifting the circle's center by a few ZIP codes each time.
For instance, Clusters 2, 3, 5, 6, 18, 26, 29, 32, and 33 are all heavily overlapping in and around Colorado.

__Ceiling saturation is concentrated in the strongest results__
| Rank band | % at spatial cap | % at temporal cap | % at both |
|---|---|---|---|
| Top 20 | 80% | 95% | 75% |
| Top 50 | 72% | 94% | 68% |
| 51–150 | 49% | 83% | 38% |
| 151–300 | 29% | 69% | 18% |
| 301–500 | 10% | 35% | 3% |

Top clusters cluster geographically in known chronically-elevated regions (Intermountain West, Alaska, rural Appalachia) — recurring centers (e.g. 87013 ×8, 84049 ×4, 88414 ×4) across non-overlapping years.
These are likely persistent regional baseline elevation, not time-bounded contagion bursts — a plain Poisson space-time scan has no way to express "always elevated" except by growing the window to the max allowed.
208/500 rows report p = 1.00E-17 exactly — a Gumbel-approximation display floor, not 208 equally-certain clusters. Use LLR rank, not p-value, below that floor.

`CriteriaForReportingSecondaryClusters=0` is the *most* restrictive option (not least) — confirmed it already preserves same-location, different-era clusters (e.g. 87013 recurring across 4 separate periods survives under current setting). No change needed here.

---

# SaTScan — Run 3

### Setup (vs. run 2)
**`SpatialAdjustmentType=2`** (Spatial Nonparametric) — adjusts expected counts to each location's own long-run average, so the test targets temporal excess above a place's own baseline rather than "this place is always hot." Directly targets the structural-vs-contagion confound above. Highest-value change.

**`TimeTrendAdjustmentType=4`**  (Time Stratified Randomization): Required by SaTScan. When applying a nonparametric spatial adjustment, the algorithm mathematically mandates a matching nonparametric temporal adjustment to prevent confounding between space and time baseline totals.

**`TimeStratifiedAdjLength=12:`** TimeStratifiedAdjLength=12: Defines the length of the temporal block used for the baseline calculation. Since time aggregation is monthly, 12 is a 1-year window.

This edit computes the expected counts against the nationwide average for each calendar year, rather than using a single average for 13-years.
12 moths effectively control for long-term secular trends—such as the gradual national rise in suicide rates over the decade, without scrubbing out the localized, month-to-month spikes that characterize contagion.

__Warning:__ Standard seasonal variation (like recurring spring peaks) could be strong enough in your dataset to trigger false-positive clusters, as a 12-month baseline smooths over the year and does not filter out within-year seasonality. (can consider `TimeStratifiedAdjLength=1` month but then you lose the long term trend adjustment)


### Run Results analysis 
**`nores_full_run3_out.txt`**: The implementation of the Nonparametric Spatial Adjustment (`SpatialAdjustmentType=2`), paired with a 12-month stratified temporal baseline (`TimeStratifiedAdjLength=12`), successfully resolved the structural confounding observed in previous runs.

**Cluster Distribution:** The geographical redundancy (e.g., repeatedly mapping the Intermountain West as a single massive cluster) has been eliminated. The model identified over 250 statistically significant clusters ($p \le 0.05$) distributed cleanly nationwide.

**Signal Isolation:** By adjusting expected counts to each location's own long-run average, the model successfully ignored "always hot" macro-regions. The resulting clusters represent true localized contagion—temporal excesses significantly above a specific ZIP code's normal baseline.

**Boundary Censoring:** A substantial portion of the highly significant top-tier clusters ($p = 1.0 \times 10^{-17}$) maxed out the maximum parameter constraints (radius $\approx 150$ km; duration $= 15$ months). 

The results confirm that localized suicide contagion exists distinct from regional baseline elevations. However, the consistent boundary censoring suggests that true contagion events can span wider and longer than the 150 km/15-month caps derived from county-level literature. I will run a sensitivty analysis expanded constraints (250 km / 21 months) is justified to capture the full lifecycle of the largest clusters.

---

# TODO SaTScan — Run4
__Using larger spacial and temporal treshold to verify boundary censoring limits__
### Setup (vs. run 3)
the values are derived from the sensitivity analysis performed by Platt. et al.
*   **`MaxSpatialSizeInDistanceFromCenter=250`** 
*   **`MaxTemporalSize=24`**

---

# TODO SaTScan — Run5
__Focus on high resolution small clusters__
### Setup (vs. run 4)
the values are derived from the sensitivity analysis performed by Platt. et al.
*   **`MaxSpatialSizeInPopulationAtRisk=0.1`** set the max size for cluster (0.1% of 322,119,726 (avg tot pop) $ \approx $ 322K people)
*   **`TimeAggregationUnits=3`** (days), **`MaxTemporalSize=30`** (30 days)
*   **`MinimumCasesInHighRateClusters=2`** it was 10, set the min case to detect the clusters
*   **`CriteriaForReportingSecondaryClusters=1`** it was 0 1 (NoCentersInOther - allows clusters to slightly overlap) or 2 (NoCentersInMostLikely)

---

# SaTScan — SpacialOnly
__Spatial only run__ 
* ### Setup (vs. run 3)
A Purely Spatial pass (`AnalysisType=1`) on the same data to map persistently-elevated baseline areas, to cross-reference against space-time clusters before interpreting them as contagion.

**`SpatialAdjustmentType=0` (None):** Disabled the spatial nonparametric adjustment. Leaving this active during a purely spatial scan forces expected cases to perfectly equal observed cases, which mathematically neutralizes the analysis (resulting in an LLR of 0.00).

**`TimeTrendAdjustmentType=0` (None):** Disabled time-stratified randomization. A purely spatial scan collapses all 13 years into a single map, meaning there is no temporal dimension left to adjust against.

**`UseDistanceFromCenterOption=n`:** Removed the 150 km distance cap. Relying solely on the default 50% population cap allows the algorithm to capture sprawling, macro-level "suicide belts" (e.g., the Intermountain West) rather than artificially chopping them into adjacent 150 km circles.

### Run Results analysis 
**`nores_full_SpacialOnly_out.txt`**
Identified massive, continent-spanning clusters (e.g., an 1,800 km cluster capturing ~32% of the US population). Mathematically confirms the Western US/Intermountain region operates as a contiguous, chronically elevated belt.It shows and validates the macro-level structural confounding, proving why the Spatial Nonparametric adjustment is mandatory for isolating contagion in space-time runs.

* ### Setup (vs. SpacialOnly)

**`UseDistanceFromCenterOption=y`**, **`MaxSpatialSizeInDistanceFromCenter=150`**: Re-applied the 150 km strict distance cap. Prevents the algorithm from engulfing half the continent, forcing it to locate the absolute hottest *localized* pockets of chronic baseline elevation.

### Run Results analysis 
**`nores_full_SpacialOnly150_out.txt`** : It shows distinct, non-overlapping regional hotspots strictly bound by the 150 km cap (e.g., severe localized pockets in Colorado, Oklahoma, South Carolina, and Oregon). It works as a high-resolution map of localized structural elevation. Cross-referencing these specific 150 km boundaries against the space-time clusters isolates ZIP codes that suffer from both a chronically high baseline *and* acute contagion bursts.


**Numbers behind the above:**

132 clusters total, real/varying LLR (top: 4,741.8, down through the hundreds).
__Top 3 clusters__: Denver/Front Range CO (148.6km, RR 2.72, pop 4.7M, LLR 4,741.8), Tulsa/OKC OK (149.2km, RR 2.58, pop 3.0M, LLR 2,661.2), Upstate SC/western NC/north GA — Appalachian border (LLR 2,547.9). Oregon hotspots appear further down the ranked list (cluster 4 onward).

**Why this pair of runs justifies `SpatialAdjustmentType=2` in the space-time runs:**:

these elevated zones aren't transient — they're sustained at RR~1.8–2.7 across the *entire* 14-year period, not a burst. An unadjusted space-time scan has no way to represent "this place is always elevated" except by growing the cylinder to the max allowed — which is exactly the ceiling-saturation pattern seen in run 2 (top clusters pegged at 150km/15mo). Adjusting nonparametrically means each location gets its own baseline rate estimated directly from its own data (no assumed functional form), so the space-time test that follows is asking "is there temporal excess above this place's own norm" rather than "is this place's rate above the national average" — the right question for isolating contagion-like bursts rather than re-detecting the baseline belt.

----

In [ ]:
#TODO: NVDRS coverage varies is high but varies by county and state over time. I need to double cehek coverage to decide what to do when i analyse overall US trends

In [ ]:
%%skip True
from src.etl.ingest import load_nvdrs

# Load raw data, forcing ZIP columns to strings
raw_df = load_nvdrs(file_key="nvdrs", data_folder="raw")
raw_df['InjuryZip'] = raw_df['InjuryZip'].astype(str).str.zfill(5)
raw_df['ResidenceZip'] = raw_df['ResidenceZip'].astype(str).str.zfill(5)
raw_df = raw_df[raw_df['IncidentCategory_c']=='Single suicide']

# Search for the ZIP in either column (.copy() prevents Pandas warnings later)
target = '06999' 

suspect_cases = raw_df[(raw_df['InjuryZip'] == target) | (raw_df['ResidenceZip'] == target)].copy()

# Convert DeathDate to proper datetime for sorting
suspect_cases['DeathDate'] = pd.to_datetime(suspect_cases['DeathDate'])

# Sort to check for temporal clustering
suspect_cases = suspect_cases.sort_values(by='DeathDate')

suspect_cases_towns = suspect_cases[['InjuryCityState', 'DeathDate']]
print(suspect_cases_towns)

target = '04067' ### all unrelated cases
suspect_cases = raw_df[(raw_df['InjuryZip'] == target) | (raw_df['ResidenceZip'] == target)].copy()

# Convert DeathDate to proper datetime for sorting
suspect_cases['DeathDate'] = pd.to_datetime(suspect_cases['DeathDate'])

# Sort to check for temporal clustering
suspect_cases = suspect_cases.sort_values(by='DeathDate')

suspect_cases_towns = suspect_cases[['InjuryCityState', 'DeathDate']]
print(suspect_cases_towns)

In [ ]:
#TODO: there are '06999' '04067' that are probably read improperly when i load the csv at some point.
#'06999' is strange, the injury fips and cities are in CT but here and there... need investigations?
#'04067' they seem to all unrelated cases... decide what to do


In [ ]:
%%skip True #run only once

from src.geospatial.filter_satscan import create_regional_satscan_files

map_path = get_data_path("zip_to_state_county")
zip_mapping = pd.read_csv(map_path, dtype={'ZIP': str})


nyc_counties = [
    "New York County",  # Manhattan
    "Kings County",     # Brooklyn
    "Queens County",    # Queens
    "Bronx County",     # Bronx
    "Richmond County"   # Staten Island
]


valid_zips_df = zip_mapping[zip_mapping['county'].isin(nyc_counties)]

summary_df,dropped_zips = create_regional_satscan_files(
    nickname_regional= "NYC_nores",
    nickname= "nores", 
    county_names= nyc_counties,
    state_names=['NY'] # necesary to avoid selecting counties with the same name from other states
)

# Display top ZIPs with the highest case burdens
display(summary_df.head(10))


#create_regional_satscan_files(
#    nickname="NYC",
#    state_names=["New York", "New Jersey", "Connecticut"]
#)

#### Check population of the ZIPs with most cases:

In [ ]:
%%skip True
### PLOTTING THE ZIP WITH THE MOST CASES
pop_file = PROJECT_ROOT / "data" / "processed" / "satscan" / "nores_full_pop.csv"
cases_file = PROJECT_ROOT / "data" / "processed" / "satscan" / "nores_nvdrs_analytical.csv"
geo_file = PROJECT_ROOT / "data" / "processed" / "satscan" / "nores_full_geo.csv"

pop_satscan = pd.read_csv(pop_file, dtype={'ZIP': str})
nvdrs_satscan = pd.read_csv(cases_file, dtype={'ZIP': str})
geo_satscan = pd.read_csv(geo_file, dtype={'ZIP': str})

import plotly.express as px

fig = px.scatter_geo(
    geo_satscan, 
    lat="Latitude", 
    lon="Longitude", 
    scope="north america" 
)

fig.update_traces(marker=dict(color="red", size=1))
fig.show()






# 2. Load the state/county mapping
map_path = get_data_path("zip_to_state_county")
zip_mapping = pd.read_csv(map_path, dtype={'ZIP': str})

# 3. Merge to recover state and county labels
pop_zip_c = pop_satscan.merge(zip_mapping, on='ZIP', how='left')

# Create the aggregated pivot table 
pop_pivot_check = pop_zip_c.pivot_table(index='ZIP', columns='Year', values='Population', aggfunc='sum')
# Reset the index so ZIP is a standard column
pop_pivot_check = pop_pivot_check.reset_index()
# Merge with a unique ZIP mapping to add state and county names
zip_mapping = pop_zip_c[['ZIP', 'state', 'county']].drop_duplicates(subset=['ZIP'])
pop_pivot_check = zip_mapping.merge(pop_pivot_check, on='ZIP', how='right')

#Calculate the percentage change across the boundary redraw (2019 to 2021)
pop_pivot_check['Jump_19_21'] = ((pop_pivot_check[2021] - pop_pivot_check[2019]) / pop_pivot_check[2019]) * 100
# 1. Get counts of each ZIP
nvdrs_zip_counts = nvdrs_satscan['DerivedZip'].value_counts()

# 2. Filter for ZIPs with a count of min_cases or more, then convert to a list
min_cases = 3
nvdrs_s_zips = nvdrs_zip_counts[nvdrs_zip_counts >= min_cases].index.tolist()

##### SELECT ONLY ZIPS wit more than min_case in NVDRS (see above)
#nvdrs_pop_pivot_check = pop_pivot_check[pop_pivot_check['ZIP'].isin(nvdrs_s_zips)]
nvdrs_pop_pivot_check = pop_pivot_check
# Filter out small-number noise (base population < 1000)
nvdrs_pop_pivot_check = nvdrs_pop_pivot_check[~((nvdrs_pop_pivot_check[2019] < 1000) | (nvdrs_pop_pivot_check[2021] < 1000))].copy()

# 2. Filter for ZIPs with a count of min_cases or more, then convert to a list
min_cases = 3
nvdrs_s_zips = nvdrs_zip_counts[nvdrs_zip_counts >= min_cases].index.tolist()
most_cases_zip = nvdrs_zip_counts.head(20).index.tolist()

#plot_zip_population_pivot(nvdrs_pop_pivot_check,most_cases_zip ,county_col='county',state_col='state')
#nvdrs_zip_counts.head(20)
#pop_pivot_check['ZIP']


#### Check worse ZIP population Changes:

In [ ]:
# Display the top 20 worst offenders
nvdrs_pop_pivot_check['Jump_19_21_abs'] = abs(nvdrs_pop_pivot_check['Jump_19_21'])
nvdrs_pop_pivot_check = nvdrs_pop_pivot_check.sort_values(by='Jump_19_21', key=abs, ascending=False)

# Grab the top 5 ZIP strings to feed into your plotting function
nvdr_zips_to_test = nvdrs_pop_pivot_check.head(10)['ZIP'].tolist()
plot_zip_population_pivot(nvdrs_pop_pivot_check,nvdr_zips_to_test,county_col='county',state_col='state')

print("Top 20 Largest 2019-2021 Population Jumps:")
nvdrs_pop_pivot_check.loc[nvdrs_pop_pivot_check.index, ['ZIP','state', 'county','Jump_19_21', 2019, 2020, 2021,]].head(8)

### Check the ZIP population change distributiion:

In [ ]:
%%skip True
# select only the zips that changed more than 30% from 19 to 21
import matplotlib.pyplot as plt

nvdrs_pop_pivot_check_large = nvdrs_pop_pivot_check[nvdrs_pop_pivot_check['Jump_19_21_abs'] > 30]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# First plot: All data
axes[0].plot(nvdrs_pop_pivot_check['Jump_19_21_abs'].reset_index(drop=True))
axes[0].set_title('Distribution of Jump_19_21_abs (All)')
axes[0].set_ylabel('Absolute Jump (2019-2021)')
axes[0].grid(True, linestyle='--', alpha=0.6)

# Second plot: Filtered data (> 30)
axes[1].plot(nvdrs_pop_pivot_check_large['Jump_19_21_abs'].reset_index(drop=True), color='orange')
axes[1].set_title('Distribution of Jump_19_21_abs (> 30)')
axes[1].set_ylabel('Absolute Jump (2019-2021)')
axes[1].grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()

#### CHECK THE NAME OF THE CENSUS VARIBALES YEAR BY YEAR:

In [ ]:
%%skip True

# -> var names for age groups population at zip level varies, and the meaning on the var varies
# ----> "NAME": "ZTCA5" and "DP05_0001E": "Population" remains the same, these are the one we use
import requests
import re
years_to_fetch=range(2010,2024)
vars_to_fetch = {
    "NAME": "ZTCA5",
    "DP05_0001E": "Population",
    "DP05_0005E": "Pop_<5",
    "DP05_0006E": "Pop_5-9",
    "DP05_0007E": "Pop_10-14",
    "DP05_0008E": "Pop_15-19",
    "DP05_0009E": "Pop_20-24",
    "DP05_0010E": "Pop_25-34",
    "DP05_0011E": "Pop_35-44",
    "DP05_0012E": "Pop_45-54",
    "DP05_0013E": "Pop_55-59",
    "DP05_0014E": "Pop_60-64",
    "DP05_0015E": "Pop_65-74",
    "DP05_0016E": "Pop_75-84",
    "DP05_0017E": "Pop_85+"
}

def get_profile_label_map(year: int, group: str, dataset: str = "acs5") -> dict:
    """{label: code} for one ACS Data Profile group/year, estimates only."""
    url = f"https://api.census.gov/data/{year}/acs/{dataset}/profile/groups/{group}.json"
    resp = requests.get(url)
    resp.raise_for_status()
    variables = resp.json()["variables"]
    return {v["label"]: code for code, v in variables.items() if re.match(r"^DP\d{2}_\d+E$", code)}

# Does DP05_0037E etc. mean the same thing every year you're pulling?
for year in years_to_fetch:
    code_to_label = {v: k for k, v in get_profile_label_map(year, "DP05").items()}
    print(year, {c: code_to_label.get(c, "MISSING/CHANGED") for c in vars_to_fetch if c != "NAME"})

### JUSTIFY DROPPING THE 2010 DECENNIAL CENSUS:

In [ ]:
%%skip True

# Mixing a 1-day headcount (2010 Decennial) with a 60-month rolling average (2011 ACS) 
# creates severe denominator shocks, especially in Group Quarters. 
# This block demonstrates the artifacts created by mixing the two APIs.

from src.etl.ingest import fetch_2010_decennial_population, fetch_census
from src.etl.transform import calc_pct_change
import pandas as pd

# 1. Fetch the exact Decennial count
decennial_2010 = fetch_2010_decennial_population()

# 2. Fetch the rolling ACS counts
acs_2011_2012 = fetch_census({"NAME": "ZTCA5", "DP05_0001E": "Population"}, range(2011, 2013), geo_level="zip")
acs_2011_2012.rename(columns={"zip code tabulation area": "ZIP"}, inplace=True)

# 3. Combine and Pivot
pop_compare = pd.concat([decennial_2010, acs_2011_2012], ignore_index=True)
pop_compare['Population'] = pd.to_numeric(pop_compare['Population'], errors='coerce')
pop_pivot = pop_compare.pivot(index='ZIP', columns='Year', values='Population')

# 4. Calculate the shock
pop_pivot['Jump_10_11_pct'] = calc_pct_change(pop_pivot, 2010, 2011)

# 5. Display the worst artifacts (ZIPs with > 1000 people and > 30% artificial jump)
artifacts = pop_pivot[(pop_pivot['Jump_10_11_pct'].abs() > 30) & (pop_pivot[2010] > 1000)]
print(f"Found {len(artifacts)} ZIPs with massive artificial population swings due to the API switch.")
print(artifacts[[2010, 2011, 'Jump_10_11_pct']].sort_values(by='Jump_10_11_pct', key=abs, ascending=False).head(10))

### Compare 2020 ACTUAL POP COUNTS from census vs 2020 1-year rolling window counts form ACS5:
the general decennal census reports the actual counts at the day of the census. it might differ form the 1-year rolling window population count usied by the ACS5 that sample the part of the population continusly.

the follwing commands show the fluctuations, that's why we use the ACS5 counts in the analysis

In [ ]:
%%skip True

# download the 2010 ACTUAL COUNTS in a given day (not the 1 year rolling window of ACS)
# from a different api link - the decennal census

from src.etl.ingest import fetch_2010_decennial_population, fetch_census
from src.etl.transform import calc_pct_change
import pandas as pd


# Fetch 2010 Decennial data and append to pop_zip
df_2010 = fetch_2010_decennial_population()


# Fetch 2011-2023 ACS5 1-year estiames
vars_to_fetch = {
    "NAME": "ZTCA5",
    "DP05_0001E": "Population"
}
years_to_fetch = range(2011, 2024) # range goes to 2023
pop_zip = fetch_census(vars_to_fetch, years_to_fetch, geo_level="zip")
pop_zip.rename(columns={"zip code tabulation area": "ZIP"}, inplace=True)
pop_zip = pop_zip.drop('state', axis=1) # will add state name and county name later
# population to numeric
pop_zip['Population'] = pd.to_numeric(pop_zip['Population'], errors='coerce') 

pop_zip = pd.concat([df_2010, pop_zip], ignore_index=True)

# Pivot data so years become columns for easy comparison
pop_pivot = pop_zip.pivot(index='ZIP', columns='Year', values='Population')

# ADD STATE AND COUNTY TO ZIP 
# -----> Approximation: some ZIPs crosses state and county lines. the major one indicated here, use just for lables

map_path = get_data_path("zip_to_state_county")
zip_mapping = pd.read_csv(map_path, dtype={'ZIP': str})

# Ensure target column is string for a safe merge, then merge
pop_pivot.index = pop_pivot.index.astype(str)
pop_pivot = pop_pivot.merge(zip_mapping, left_index=True, right_on='ZIP', how='left')


# Calculate Year-over-Year fluctuations
pop_pivot['pct_change_10_11'] = calc_pct_change(pop_pivot, 2010, 2011)
pop_pivot['pct_change_11_12'] = calc_pct_change(pop_pivot, 2011, 2012)
pop_pivot['pct_change_12_13'] = calc_pct_change(pop_pivot, 2012, 2013)

# Filter for suspect fluctuations 
threshold_pct = 15
min_pop = 1000

suspect_10_11 = pop_pivot[(pop_pivot['pct_change_10_11'].abs() > threshold_pct) & (pop_pivot[2010] > min_pop)]
suspect_11_12 = pop_pivot[(pop_pivot['pct_change_11_12'].abs() > threshold_pct) & (pop_pivot[2011] > min_pop)]
suspect_12_13 = pop_pivot[(pop_pivot['pct_change_12_13'].abs() > threshold_pct) & (pop_pivot[2012] > min_pop)]

# Review the damage
print(f"Artifacts 2010-2011: {len(suspect_10_11)} ZIPs")
print(f"Artifacts 2011-2012: {len(suspect_11_12)} ZIPs")
print(f"Artifacts 2012-2013: {len(suspect_12_13)} ZIPs")

print("\nTop 10 Largest Fluctuations (2010 to 2011):")
suspect_10_11[[2010, 2011, 'pct_change_10_11', 'county', 'state']].sort_values(by='pct_change_10_11', ascending=False).head(10)